# Ball Detector Training: Merged Dataset

**Purpose:** Fine-tune YOLOv8x as a single-class ball detector on the merged ball dataset.  
**Inputs:** Roboflow dataset `nba-ball-detection-merged` v1; `yolov8x.pt` base weights;
`training/augment_ball_images.py`.  
**Outputs:** the `runs/detect/ball_train_merged/` training run and its `best.pt` checkpoint.  
**Backs:** no results/ artefact; the checkpoint was superseded in production, and this
notebook remains the hyperparameter and convention basis for
`train_ball_fy4c2_replication.ipynb`.

The first cell pins the working directory to the repo root, so every relative path
resolves the same way wherever the kernel was launched.

In [1]:
import os
from pathlib import Path

# Fixes the double-nested `training/training/...` bug at the source:
# this notebook lives in training/, so a kernel launched there resolves every
# relative path one level too deep. Pin the working directory to the repo root.
if Path.cwd().name == 'training':
    os.chdir('..')

repo_root = Path.cwd()
assert (repo_root / 'training').is_dir() and (repo_root / 'basketball').is_dir(), (
    f'Unexpected working directory: {repo_root}. '
    f'Launch this notebook from the repo root or from training/.'
)
print(f'Working directory: {repo_root}')

Working directory: /home/jovyan/nba-video-analytics


## 1. Environment check

Confirms the GPU and the Ultralytics install before anything else runs.

In [2]:
import torch
import ultralytics

print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'Ultralytics: {ultralytics.__version__}')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA not available — verify kernel is set to Python (Basketball Analytics) before proceeding.')

PyTorch: 2.4.0+cu124 | CUDA: True | GPU: NVIDIA A40
Ultralytics: 8.4.62


## 2. Download dataset

Downloads the **nba-ball-detection-merged** dataset (v1, single class: `ball`)
from Roboflow in YOLOv8 format. This merged dataset supplements the original
ball images with additional ball-labelled frames for this training run.

Export your personal Roboflow API key as `ROBOFLOW_API_KEY` in the kernel environment
before running the download cell below. **Never hardcode or commit a real key.**

In [11]:
from pathlib import Path

dataset_dir = Path('training/ball-detection-merged')
dataset_dir.mkdir(parents=True, exist_ok=True)
print(f'Dataset directory: {dataset_dir.resolve()}')

Dataset directory: /home/jovyan/nba-video-analytics/training/ball-detection-merged


In [12]:
# Reads the Roboflow API key from the ROBOFLOW_API_KEY environment variable:
# export it in the kernel environment before running; never hardcode a key here.
# Mirrors the "YOLOv8" Jupyter export snippet on the Roboflow version page;
# fill in your workspace slug from that same snippet.
import os

from roboflow import Roboflow

rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
project = rf.workspace('hanad-ali').project('nba-ball-detection-merged-4feju')
version = project.version(1)
dataset = version.download('yolov8', location=str(dataset_dir), overwrite=True)

print(f'Download complete: {dataset.location}')

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to training/ball-detection-merged in yolov8:: 100%|██████████| 4017/4017 [00:00<00:00, 5941.09it/s]


Download complete: /home/jovyan/nba-video-analytics/training/ball-detection-merged


## 3. Dataset verification

Confirms the downloaded dataset has the expected class name, image counts, and label distribution.

In [13]:
import yaml
from pathlib import Path

# Gate cell: a human compares this output against the Roboflow version page
# before training proceeds. Every count is read directly from the downloaded
# label files, not from any docstring or comment.

data_yaml_path = dataset_dir / 'data.yaml'
with open(data_yaml_path) as f:
    data_config = yaml.safe_load(f)

class_names = data_config['names']
print('Dataset configuration:')
print(f'  Classes (nc): {data_config["nc"]}')
print(f'  Class names:  {class_names}')

assert data_config['nc'] == 1, f'Expected exactly 1 class, got {data_config["nc"]}.'
assert [name.lower() for name in class_names] == ['ball'], (
    f'Expected a single class named "ball", got {class_names}.'
)
print('  -> single "ball" class confirmed.')

splits = ['train', 'valid', 'test']
total_images = 0
total_ball_annotations = 0
total_bad_labels = 0

print('\nPer-split counts (read from label files):')
for split in splits:
    images_dir = dataset_dir / split / 'images'
    labels_dir = dataset_dir / split / 'labels'

    image_count = sum(
        1 for image_path in images_dir.glob('*')
        if image_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}
    ) if images_dir.is_dir() else 0

    ball_annotations = 0
    bad_labels = 0
    if labels_dir.is_dir():
        for label_path in labels_dir.glob('*.txt'):
            for line in label_path.read_text().splitlines():
                if not line.strip():
                    continue
                if int(line.split()[0]) == 0:
                    ball_annotations += 1
                else:
                    bad_labels += 1

    total_images += image_count
    total_ball_annotations += ball_annotations
    total_bad_labels += bad_labels
    print(f'  {split:5s}: {image_count:5d} images | {ball_annotations:5d} ball annotations | {bad_labels} non-ball labels')

print(f'\nTotals: {total_images} images | {total_ball_annotations} ball annotations across all splits.')

assert total_bad_labels == 0, (
    f'Found {total_bad_labels} labels with class id != 0 — dataset is not single-class ball.'
)
print('Label integrity confirmed: 0 labels with class id != 0.')

Dataset configuration:
  Classes (nc): 1
  Class names:  ['ball']
  -> single "ball" class confirmed.

Per-split counts (read from label files):
  train:  1605 images |  1604 ball annotations | 0 non-ball labels
  valid:   201 images |   201 ball annotations | 0 non-ball labels
  test :   200 images |   200 ball annotations | 0 non-ball labels

Totals: 2006 images | 2005 ball annotations across all splits.
Label integrity confirmed: 0 labels with class id != 0.


## 4. Augment training split

Runs `augment_ball_images.py` to triple the training split (2 offline augmented
copies per image: rotation, brightness/contrast, motion blur, and scale jitter,
each applied independently at random under a fixed seed of 42, and bbox-aware). Validation and test splits
are never augmented. The cell asserts the expected x3 image and annotation
arithmetic before training proceeds.

In [20]:
import re
import subprocess
import sys
from pathlib import Path

train_images_dir = dataset_dir / 'train' / 'images'
train_labels_dir = dataset_dir / 'train' / 'labels'


def count_images(images_dir: Path) -> int:
    return sum(
        1 for image_path in images_dir.glob('*')
        if image_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}
    )


def count_ball_annotations(labels_dir: Path) -> int:
    total = 0
    for label_path in labels_dir.glob('*.txt'):
        for line in label_path.read_text().splitlines():
            if line.strip() and int(line.split()[0]) == 0:
                total += 1
    return total


already_augmented = list(train_images_dir.glob('*_aug*'))
if already_augmented:
    raise RuntimeError(
        f'{len(already_augmented)} augmented images already present in the train split. '
        f'Re-download the dataset before augmenting — re-running would compound x3 on x3.'
    )

images_before = count_images(train_images_dir)
annotations_before = count_ball_annotations(train_labels_dir)

result = subprocess.run(
    [sys.executable, 'training/augment_ball_images.py',
     '--dataset-dir', str(dataset_dir), '--num-copies', '2', '--seed', '42'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Augmentation failed — see stderr above.')

images_after = count_images(train_images_dir)
annotations_after = count_ball_annotations(train_labels_dir)

print(f'Train images:     {images_before} -> {images_after} (x{images_after / images_before:.1f})')
print(f'Ball annotations: {annotations_before} -> {annotations_after} (x{annotations_after / annotations_before:.1f})')

match = re.search(r'Found (\d+) training images containing the ball class', result.stdout)
if match is None:
    raise RuntimeError('Could not parse ball-image count from augmentation script output.')
n_ball_images = int(match.group(1))

expected_images_after = images_before + n_ball_images * 2
assert images_after == expected_images_after, (
    f'Expected {expected_images_after} images ({images_before} original + '
    f'{n_ball_images} ball-containing images x2 copies), got {images_after}.'
)
assert annotations_after == annotations_before * 3, (
    f'Expected x3 ball annotations ({annotations_before * 3}), got {annotations_after}.'
)

print(f'\nAugmentation verified: {n_ball_images} ball-containing images expanded x3 '
      f'(2 copies per image); {images_before - n_ball_images} image(s) with no ball '
      f'annotation left untouched; valid/test untouched.')

Before augmentation: 1605 images, 1604 ball annotations.
Found 1604 training images containing the ball class.
After augmentation: 4813 images, 4812 ball annotations.
Train split expanded from 1605 to 4813 images (x3.0) and from 1604 to 4812 ball annotations (x3.0).

Train images:     1605 -> 4813 (x3.0)
Ball annotations: 1604 -> 4812 (x3.0)

Augmentation verified: 1604 ball-containing images expanded x3 (2 copies per image); 1 image(s) with no ball annotation left untouched; valid/test untouched.


## 5. Train YOLOv8x

Fine-tunes YOLOv8x on the augmented merged ball dataset. Hyperparameters match
the player detector's: 100-epoch ceiling with early stopping (patience=30),
batch size 32 on the A40 GPU, 640x640 input resolution, and Ultralytics'
default seed=0. The run name `ball_train_merged` names the dataset it was
trained on. The IProgress and IOPub rate messages in the output are Jupyter
display notices; they do not affect training.

In [17]:
from ultralytics import YOLO
from pathlib import Path

data_yaml = str((dataset_dir / 'data.yaml').resolve())

model = YOLO('yolov8x.pt')

# No explicit project= argument: passing one causes a path-doubling
# bug. Ultralytics defaults the output to runs/detect/ball_train_merged/.
results = model.train(
    data=data_yaml,
    epochs=100,
    patience=30,
    batch=32,
    imgsz=640,
    name='ball_train_merged',
)

print(f'\nTraining complete. Best weights: {results.save_dir}/weights/best.pt')

New https://pypi.org/project/ultralytics/8.4.89 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.11.15 torch-2.4.0+cu124 CUDA:0 (NVIDIA A40, 45619MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/nba-video-analytics/training/ball-detection-merged/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momen

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 719.7±368.3 MB/s, size: 87.2 KB)
val: Scanning /home/jovyan/nba-video-analytics/training/ball-detection-merged/valid/labels... 201 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 201/201 188.1it/s 1.1s0.2s
val: New cache created: /home/jovyan/nba-video-analytics/training/ball-detection-merged/valid/labels.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.0005), 103 bias(decay=0.0)
Plotting labels to /home/jovyan/nba-video-analytics/runs/detect/ball_train_merged/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dat

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



     63/100      23.3G     0.9965      0.469     0.8238         13        640: 100% ━━━━━━━━━━━━ 151/151 1.3it/s 1:540.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.1it/s 1.9s0.9s
                   all        201        201      0.983       0.92      0.969      0.611

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     64/100      22.9G     0.9931     0.4691     0.8254         13        640: 100% ━━━━━━━━━━━━ 151/151 1.3it/s 1:540.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.2it/s 1.8s0.9s
                   all        201        201      0.978        0.9      0.949      0.601

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     65/100      23.2G     0.9793     0.4674     0.8202         13        640: 100% ━━━━━━━━━━━━ 151/151 1.3it/s 1:540.6ss
                 Class     Images  In

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



     94/100      22.9G     0.7137     0.3287     0.8015         13        640: 100% ━━━━━━━━━━━━ 151/151 1.3it/s 1:530.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.1it/s 1.9s0.9s
                   all        201        201      0.984      0.935      0.959      0.618

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     95/100      23.2G     0.7038     0.3227     0.8033         13        640: 100% ━━━━━━━━━━━━ 151/151 1.3it/s 1:540.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.2it/s 1.8s0.9s
                   all        201        201      0.984      0.925      0.951      0.615

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     96/100      22.9G     0.6782     0.3164     0.7952         13        640: 100% ━━━━━━━━━━━━ 151/151 1.3it/s 1:530.5ss
                 Class     Images  In

## 6. Per-class metrics

Reports final validation metrics for the ball class. Metrics are read directly
from the training run's `results.csv` on disk (located via glob), not from an
in-memory `results` variable, so they cannot be stale from a previous kernel
session.

In [18]:
import glob
import pandas as pd
from pathlib import Path

run_dirs = glob.glob('runs/**/ball_train_merged*/', recursive=True)
if not run_dirs:
    raise FileNotFoundError('No ball_train_merged run directory found under runs/.')

latest_run = Path(sorted(run_dirs)[-1])
results_csv = latest_run / 'results.csv'

history = pd.read_csv(results_csv)
history.columns = history.columns.str.strip()

# best.pt is chosen by Ultralytics fitness (weighted mAP); approximate it here
# by the epoch with the highest mAP50-95 and report that epoch's metrics.
best_row = history.loc[history['metrics/mAP50-95(B)'].idxmax()]

print(f'Metrics read from: {results_csv}')
print(f'Epochs logged:     {len(history)} (best epoch: {int(best_row["epoch"])})')
print('\nFinal validation metrics (best epoch, matches best.pt):')
print(f'  Ball precision:  {best_row["metrics/precision(B)"]:.4f}')
print(f'  Ball recall:     {best_row["metrics/recall(B)"]:.4f}')
print(f'  Ball mAP50:      {best_row["metrics/mAP50(B)"]:.4f}')
print(f'  Ball mAP50-95:   {best_row["metrics/mAP50-95(B)"]:.4f}')

Metrics read from: runs/detect/ball_train_merged/results.csv
Epochs logged:     100 (best epoch: 87)

Final validation metrics (best epoch, matches best.pt):
  Ball precision:  0.9729
  Ball recall:     0.9403
  Ball mAP50:      0.9664
  Ball mAP50-95:   0.6347


## 7. Locate best checkpoint

Confirms the location and size of the best checkpoint. Download this file, rename
it to `ball.pt`, and place it in `models/` on the local machine. The recursive
glob is kept as a safety net in case the output path is nested unexpectedly.

In [19]:
import glob
from pathlib import Path

checkpoints = glob.glob('runs/**/ball_train_merged*/weights/best.pt', recursive=True)

if checkpoints:
    best_pt = sorted(checkpoints)[-1]
    size_bytes = Path(best_pt).stat().st_size
    size_mb = size_bytes / (1024 * 1024)
    print(f'Best checkpoint: {best_pt}')
    print(f'File size:       {size_mb:.1f} MB ({size_bytes:,} bytes)')
    print(f'\nNext step: download {best_pt}, rename to ball.pt, place in models/')
else:
    print('No checkpoint found — check runs/ for the training output directory.')

Best checkpoint: runs/detect/ball_train_merged/weights/best.pt
File size:       130.4 MB (136,729,267 bytes)

Next step: download runs/detect/ball_train_merged/weights/best.pt, rename to ball.pt, place in models/


## 8. Outcome

The run produced `runs/detect/ball_train_merged/weights/best.pt`, deployed at the
time as `models/ball.pt`. The fy4c2 replication checkpoint later superseded it under
that same name; this notebook remains the hyperparameter and convention basis for
`train_ball_fy4c2_replication.ipynb`.